# Fidelity models on the `LinearMap` abstraction

This notebook summarizes the redesign of the fidelity-model stack around a single
linear-algebra abstraction — `LinearMap` — and the sparse, label-indexed containers it is
built on. We work bottom-up:

1. **The design at a high level** — what changed and why.
2. **The `math` layer** — `IndexedVector`, `IndexedMatrix`, and `LinearMap` / `ComposedLinearMap`.
3. **The fidelity models** — how `FidelityModel`, `CompleteFidelityModel`, `PauliLindbladModel`,
   and `ComposedFidelityModel` sit on top of `LinearMap`.
4. **Downstream** — the `analysis` and `visualizations` updates that consume the abstraction.

## 1. At a high level

A **fidelity model** is a linear parameterization of the log fidelities of a gate set: the matrix
$A$ for which $-\log(f) = A\,r$, where $f$ is the vector of gate-transition fidelities and $r$ is
the model parameter vector.

The redesign expresses this directly by defining a `LinearMap` abstraction in the `math` folder:
- It defines an implicitly defined matrix in terms of an input and output vector space (specified by a `ParameterSpace` class), and a mechanism for querying rows
- They can be composed, and concrete submatrices and linear algebra can be performed utilizing the `IndexedVector` and `IndexedMatrix` classes

A `FidelityModel` is defined as a `LinearMap`: 
- The output space is a `LogFidelitySpace` (specific parameter space)
- Has various special functions based on "fidelity" modelling (e.g. `row_from_path`)

Linear maps can be composed, creating `ComposedLinearMap` (or `ComposedFidelityModel` if the last map is a `FidelityModel`).
- `FidelityMixer` is deleted - it will now just be a special kind of `FidelityModel` (that maps log-fidelities -> log-fidelities) that can be composed with another `FidelityModel`
- Composition is very nice mathematically but poses some interesting technical issues: e.g. if there is special functionality for a `PauliLindbladModel` (e.g. `to_pauli_lindblad_maps(model_data)`) but it's inside of a `ComposedFidelityModel`, then the `PauliLindbladModel` needs to be identified in the chain and the model data needs to be propagated up to the `PauliLindbladModel` (and anything after is ignored) (more later)

## 2. `math` module

Introduces `LinearMap` and `ParameterSpace`.

In [1]:
# A ParameterSpace describes a vector space with arbitrary basis-index labels.
# EnumeratedParameterSpace is the simplest concrete one: an explicit finite set of indices.
from qiskit_noise_learning.math import EnumeratedParameterSpace

space = EnumeratedParameterSpace(frozenset({"i0", "i1", "i2"}))

In [2]:
space.dim

3

In [3]:
"i0" in space

True

In [4]:
sorted(space)

['i0', 'i1', 'i2']

In [5]:
# A LinearMap is an implicitly-defined matrix: subclasses only implement row(output_index),
# returning the sparse row as an IndexedVector over the input space. Everything else is derived.
from qiskit_noise_learning.math import IndexedVector, LinearMap


class DictLinearMap(LinearMap):
    # a linear map defined by explicit sparse rows
    def __init__(self, rows):
        self._rows = rows
        inputs = frozenset(idx for row in rows.values() for idx in row)
        super().__init__(
            input_space=EnumeratedParameterSpace(inputs),
            output_space=EnumeratedParameterSpace(frozenset(rows)),
        )

    def row(self, output_index):
        return self._rows[output_index]


# o0 = i0 + i1,  o1 = 2 * i1
example = DictLinearMap(
    {
        "o0": IndexedVector({"i0": 1.0, "i1": 1.0}),
        "o1": IndexedVector({"i1": 2.0}),
    }
)

print("input space   :", sorted(example.input_space))
print("output space  :", sorted(example.output_space))
print("row(o0)       :", dict(example.row("o0")))
# forward dot product (M x)[o0]
print("evaluate(o0)  :", example.evaluate("o0", {"i0": 5.0, "i1": 7.0}))
# transpose action: output-space vector -> input-space vector
print("left_multiply :", dict(example.left_multiply(IndexedVector({"o0": 1.0, "o1": 1.0}))))

input space   : ['i0', 'i1']
output space  : ['o0', 'o1']
row(o0)       : {'i0': 1.0, 'i1': 1.0}
evaluate(o0)  : 12.0
left_multiply : {'i0': 1.0, 'i1': 3.0}


`LinearMap`s can be composed

In [7]:
# 'a @ b' means a applied after b. Composition is generic and flattens into a single chain;
# the composed row is obtained by folding through the chain.
second = DictLinearMap({"p": IndexedVector({"o0": 1.0, "o1": 1.0})})

chain = second @ example  # 'second' applied after 'example'
print("type   :", type(chain).__name__)
print("n maps :", len(chain.maps))
print("row(p) :", dict(chain.row("p")))

type   : ComposedLinearMap
n maps : 2
row(p) : {'i0': 1.0, 'i1': 3.0}


In [9]:
# A LinearMap can be materialized as an IndexedMatrix over an explicit set of output rows
# (the full output space is in general too large to enumerate).
import numpy as np

from qiskit_noise_learning.math import IndexedMatrix

matrix = example.to_indexed_matrix(["o0", "o1"])

# index-aware matrix-vector product reproduces forward application
x = IndexedVector({"i0": 5.0, "i1": 7.0})
print("matrix @ x :", dict(matrix @ x))

# transpose (.T) and label-aware matmul make covariance propagation read directly: M cov M^T
cov = IndexedMatrix.from_index_lists(
    row_indices=["i0", "i1"], column_indices=["i0", "i1"], data=np.diag([2.0, 3.0])
)
print("M cov M^T  :\n", (matrix @ cov @ matrix.T).data)

matrix @ x : {'o0': 12.0, 'o1': 14.0}
M cov M^T  :
 [[ 5.  6.]
 [ 6. 12.]]


## 3. The fidelity models

A `FidelityModel` is a `LinearMap` whose **output space is a `LogFidelitySpace`** — the coordinates
are log fidelities, *indexed* by `FidelityIndex` objects (the indices are the labels, not the
parameters). Concrete models implement `row`; the base class supplies the fidelity-domain
conveniences and composition.

### A gate set

First a small gate set: a `CZ` Clifford layer plus pure preparation (`P`) and measurement (`M`).

In [10]:
from qiskit.circuit.library import CZGate
from qiskit.quantum_info import Clifford, QubitSparsePauli

from qiskit_noise_learning.gate_sets import ModelGate, ModelGateSet

gate_set = ModelGateSet(2)
gate_set.add_gate(
    ModelGate("CZ", [((0, 1), Clifford(CZGate()))], qubit_idxs=[0, 1], latex_str=r"\mathrm{CZ}")
)
gate_set.add_gate(ModelGate("P", qubit_idxs=range(2), prep_idxs=range(2)))
gate_set.add_gate(ModelGate("M", qubit_idxs=range(2), meas_idxs=range(2)))
gate_set

Name,Num Qubits,Gate Edges,Prep,Meas,Idling
CZ,2,0_1,—,—,—
P,2,—,"0, 1",—,—
M,2,—,—,"0, 1",—


The existing model class interfaces are not really changed.

In [11]:
from qiskit_noise_learning.math import LinearMap
from qiskit_noise_learning.models import (
    CompleteFidelityModel,
    PauliLindbladModel,
)

complete = CompleteFidelityModel(gate_set)
plm = PauliLindbladModel.k_local(gate_set, k=2)

In [12]:
plm.output_space

Has an associated gate set. Perhaps this also needs to have some pared down version of the gate set properties required to know what valid fidelities are.

E.g. `GateProperties` and `GetSetProperties`. Had discussed this as an option for `FidelityIndex`/`ApplyGate` refactor. Alternatively, we could just do a `from_gate_set` method that grabs the required properties of each gate internally.

In [13]:
plm.output_space.gate_set

Name,Num Qubits,Gate Edges,Prep,Meas,Idling
CZ,2,0_1,—,—,—
P,2,—,"0, 1",—,—
M,2,—,—,"0, 1",—


In [14]:
plm.output_space.dim

36

In [15]:
plm.input_space

In [16]:
plm.input_space._indices

frozenset({GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_1 X_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_1 Y_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_1 Z_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_1>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_1 X_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_1 Y_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_1 Z_0>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_1>),
           GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 q

Existing, fidelity-index specific methods still exist

In [17]:
from qiskit_noise_learning.sequences import FidelityIndex, Path

fi = FidelityIndex.from_gate(
    gate=gate_set["CZ"],
    pauli=QubitSparsePauli.from_sparse_label(("X", [0]), num_qubits=2),
)
path = Path(start_fragment=[], repeatable_fragment=[fi, fi], end_fragment=[], depth=4)
plm.row_from_path(path)

{GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Z_0>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_0>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_1>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_1>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Z_1 Z_0>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Z_1 Y_0>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: X_1 X_0>): 16.0,
 GeneratorIndex(gate_name='CZ', generator=<QubitSparsePauli on 2 qubits: Y_1 X_0>): 16.0}

### Composition

`FidelityModel`s can be composed like other linear maps

In [18]:
complete @ plm

In [19]:
(complete @ plm).maps

## 4. Downstream: `analysis` and `visualizations`

With this setup, some technical challenges:
- A function like `PauliLindbladModel.to_pauli_lindblad_maps` needs to be rethought: a user could have a `ComposedFidelityModel` with a `PauliLindbladModel` in the chain
- Similarly, the latex symbol generation functions were dependent on the type of model

Here, we need to modify this functionality to exist with this framework.

### Propagating fitted data — `analysis.propagate_model_data`

For defining a `to_pauli_lindblad_maps`:
- If the input map is a `PauliLindbladModel`, fine
- If it is a `ComposedFidelityModel`:
    - Find a `PauliLindbladModel` in the chain
    - Propagate the fitted data through any linear maps preceding the `PauliLindbladModel`
    - Call the `PauliLindbladModel.to_pauli_lindblad_maps` on the result of the above.

In [20]:
from qiskit_noise_learning.analysis import propagate_model_data
from qiskit_noise_learning.data import ModelData

md_in = ModelData.from_arrays(
    parameter_indices=["i0", "i1"],
    parameter_values=np.array([5.0, 7.0]),
    covariance=np.diag([2.0, 3.0]),
    time_lbs=np.array(["2024-01-01", "2024-01-03"], dtype="datetime64[s]"),
    time_ubs=np.array(["2024-01-02", "2024-01-05"], dtype="datetime64[s]"),
)

# reuse the explicit map from section 2: o0 = i0 + i1, o1 = 2 * i1
md_out = propagate_model_data(example, md_in, ["o0", "o1"])

print(
    "values :",
    dict(zip(md_out.dataset.coords["parameter"].values, md_out.dataset["parameter_values"].values)),
)
print("cov    :\n", md_out.dataset["covariance"].values)

values : {'o0': np.float64(12.0), 'o1': np.float64(14.0)}
cov    :
 [[ 5.  6.]
 [ 6. 12.]]


### `analysis.to_pauli_lindblad_maps`

In [21]:
from qiskit_noise_learning.analysis import to_pauli_lindblad_maps

indices = list(plm.input_space)
plm_data = ModelData.from_arrays(
    parameter_indices=indices,
    parameter_values=np.full(len(indices), 0.01),
    covariance=np.eye(len(indices)),
    time_lbs=np.empty(len(indices), dtype="datetime64[us]"),
    time_ubs=np.empty(len(indices), dtype="datetime64[us]"),
)


class IdentityMap(LinearMap):
    def __init__(self, space):
        super().__init__(input_space=space, output_space=space)

    def row(self, output_index):
        return IndexedVector({output_index: 1.0})


standalone = to_pauli_lindblad_maps(plm, plm_data)
composed_plm = plm.pre_compose(IdentityMap(plm.input_space))
through_chain = to_pauli_lindblad_maps(composed_plm, plm_data)

print("standalone gates    :", sorted(standalone))
print("through a chain gates:", sorted(through_chain))

standalone gates    : ['CZ']
through a chain gates: ['CZ']


### Latex `visualizations`

Similarly, now a function that inspects a `FidelityModel` and makes decisions on how to render based on its contents.

In [25]:
from qiskit_noise_learning.visualizations import fidelity_index_latex_str

print("index (formula)  :", fidelity_index_latex_str(fi, plm, format="formula"))
print("index  (transition):", fidelity_index_latex_str(fi, complete, format="formula"))

index (formula)  : f^{\mathrm{CZ}}_{X_{0} Z_{1}}
index  (transition): f^{\mathrm{CZ}}(X_{0})
